# Second draws --- does block34 reproduce? (2026-08-29)

The sweep has five arms and, since today, a **measured** noise floor. The floor
is the point of this notebook: two independent unseeded draws of the `frozen`
specification --- same data, same folds, same seed, nothing changed --- differ
by **+0.0035** in the paired sixteen-station mean at **t = +1.34**, with one
station moving 0.0256.

A comparison of one specification against *itself* produced most of a
conventional significance threshold. So the paper cannot yet say what
block34's +0.0109 at t = +2.45 means, because block34 was run once.

| arm | what it settles | cost |
|---|---|---|
| `nopogonias_rep2` | whether the -0.0462 recall cost (the only effect above the floor) reproduces | minutes/fold, cached features |
| `frozen_rep3` | turns the floor from one number into a range | minutes/fold, cached features |
| `block34_rep2` | **the decisive one**: is +0.0109 the model or the draw? | ~25 min/fold on a T4 |

Cheapest first on purpose: a session that dies after two hours should leave
finished arms behind rather than three half-swept ones.

**Safe to interrupt.** Every fold syncs to Drive as it completes and re-running
the run cell picks up from whatever is already there, so closing the lid costs
the fold in flight and nothing before it.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q -b v13-honest-labels https://github.com/Mo119m/primates-sound-detection /content/repo
%cd /content/repo
!git log --oneline -1
U = '/content/drive/MyDrive/primates-sound-detection'
!mkdir -p /content/dataF && cp -n {U}/v13_images.npy {U}/v13_index.csv {U}/manifest.csv /content/dataF/
!ls -la /content/dataF

In [ ]:
# 22,169 rows is the build every number in the paper comes from; 21,120 is
# the one before it. Assert rather than hope.
import pandas as pd
i = pd.read_csv('/content/dataF/v13_index.csv')
print(len(i), 'rows')
assert len(i) == 22169, 'wrong dataset on Drive'
print(i.label.value_counts().to_dict())

In [ ]:
# The time gate needs its lookup table. Without it every fold trains fine and
# comes out missing the four columns the comparison reads -- which is what
# happened on 24 August and cost seven folds. Fail here, where it is free.
import os
import pandas as pd
G = '/content/repo/data/outputs/auto_cleanup/review_gate_table.csv'
assert os.path.exists(G), 'no review gate table in the clone -- pull the branch again'
g = pd.read_csv(G)
print(len(g), 'rows,', list(g.columns))
assert len(g) == 6189, 'wrong gate table'
assert set(g.columns) == {'file', 'timestamp', 'start_s'}, 'unexpected columns'
print('time gate has its clock')

In [ ]:
A = '--manifest /content/dataF/manifest.csv --index /content/dataF/v13_index.csv --images /content/dataF/v13_images.npy --cache /content/dataF/v13_features.npy'
!python scripts/train_v13_loso.py --prepare-cache-only --overwrite {A} --out /content/warm.csv --run-metadata /content/cacheF.run.json

In [ ]:
# Three arms, sixteen folds each, synced fold by fold.
# Re-run this cell after any disconnect: finished folds are skipped.
!python colab/run_replicates.py

In [ ]:
# Read the arms back without retraining, once they exist. Each replicate is
# printed beside its own first draw; the draw-to-draw line is the one that
# decides whether the original measured the model or the draw.
import sys
sys.argv = ['x']
sys.path.insert(0, '/content/repo/colab')
import run_replicates
run_replicates.summarise()